In [7]:
import duckdb
import pandas as pd

# 1. Connect to DuckDB (creates a local file-based database)
conn = duckdb.connect('options_analytics.duckdb')
print("Connected to DuckDB.")

# 2. Load the CSV into a staging table
# Replace 'nse_data.csv' with the exact name of your downloaded Kaggle file
csv_file_path = 'data_analytics.csv'

print("Loading Kaggle CSV into database...")
conn.execute(f"""
    CREATE OR REPLACE TABLE raw_trades AS 
    SELECT * FROM read_csv_auto('{csv_file_path}');
""")

Connected to DuckDB.
Loading Kaggle CSV into database...


In [8]:
# Query to view the first 5 rows of your data
query = "SELECT * FROM raw_trades LIMIT 5;"

# Execute the query and display it nicely as a Pandas DataFrame
df_preview = conn.execute(query).df()
display(df_preview)

,column00,INSTRUMENT,SYMBOL,EXPIRY_DT,STRIKE_PR,OPTION_TYP,OPEN,HIGH,LOW,CLOSE,SETTLE_PR,CONTRACTS,VAL_INLAKH,OPEN_INT,CHG_IN_OI,TIMESTAMP
0,160393,FUTIDX,BANKNIFTY,29-Aug-2019,0.0,XX,28805.65,28924.00,28140.55,28499.30,28499.30,214569,1225914.96,1675780,234640,01-AUG-2019
1,160394,FUTIDX,BANKNIFTY,26-Sep-2019,0.0,XX,28926.40,29030.55,28251.70,28611.45,28611.45,2484,14245.95,51400,-80,01-AUG-2019
2,160395,FUTIDX,BANKNIFTY,31-Oct-2019,0.0,XX,29000.00,29105.00,28355.55,28699.05,28699.05,598,3434.43,9460,4860,01-AUG-2019
3,160396,FUTIDX,NIFTY,29-Aug-2019,0.0,XX,11098.40,11098.40,10901.10,11015.35,11015.35,199881,1650955.24,19001400,1339200,01-AUG-2019
4,160397,FUTIDX,NIFTY,26-Sep-2019,0.0,XX,11136.35,11145.20,10955.00,11066.60,11066.60,5283,43841.57,893625,66750,01-AUG-2019


In [9]:
conn.execute("""
    ALTER TABLE raw_trades ADD COLUMN IF NOT EXISTS exchange VARCHAR;
    UPDATE raw_trades SET exchange = 'NSE';
    
    -- Mocking some data for MCX (Gold) and BSE for the cross-exchange query
    UPDATE raw_trades SET exchange = 'MCX', symbol = 'GOLD' WHERE symbol = 'RELIANCE' AND random() < 0.1;
    UPDATE raw_trades SET exchange = 'BSE' WHERE symbol = 'INFY' AND random() < 0.1;
""")

In [10]:
# Query to view the first 5 rows of your data
query = "SELECT * FROM raw_trades LIMIT 5;"

# Execute the query and display it nicely as a Pandas DataFrame
df_preview = conn.execute(query).df()
display(df_preview)


,column00,INSTRUMENT,SYMBOL,EXPIRY_DT,STRIKE_PR,OPTION_TYP,OPEN,HIGH,LOW,CLOSE,SETTLE_PR,CONTRACTS,VAL_INLAKH,OPEN_INT,CHG_IN_OI,TIMESTAMP,exchange
0,160393,FUTIDX,BANKNIFTY,29-Aug-2019,0.0,XX,28805.65,28924.00,28140.55,28499.30,28499.30,214569,1225914.96,1675780,234640,01-AUG-2019,NSE
1,160394,FUTIDX,BANKNIFTY,26-Sep-2019,0.0,XX,28926.40,29030.55,28251.70,28611.45,28611.45,2484,14245.95,51400,-80,01-AUG-2019,NSE
2,160395,FUTIDX,BANKNIFTY,31-Oct-2019,0.0,XX,29000.00,29105.00,28355.55,28699.05,28699.05,598,3434.43,9460,4860,01-AUG-2019,NSE
3,160396,FUTIDX,NIFTY,29-Aug-2019,0.0,XX,11098.40,11098.40,10901.10,11015.35,11015.35,199881,1650955.24,19001400,1339200,01-AUG-2019,NSE
4,160397,FUTIDX,NIFTY,26-Sep-2019,0.0,XX,11136.35,11145.20,10955.00,11066.60,11066.60,5283,43841.57,893625,66750,01-AUG-2019,NSE


Top 10 Symbols by OI Change

In [12]:
query_1 = """
    WITH DailyOI AS (
        SELECT 
            symbol,
            exchange,
            -- Use strptime to parse the '09-AUG-2019' format, then cast to DATE
            strptime(timestamp, '%d-%b-%Y')::DATE as trade_date,
            SUM(open_int) as total_oi,
            LAG(SUM(open_int)) OVER (
                PARTITION BY symbol, exchange 
                ORDER BY strptime(timestamp, '%d-%b-%Y')::DATE
            ) as prev_oi
        FROM raw_trades
        GROUP BY symbol, exchange, strptime(timestamp, '%d-%b-%Y')::DATE
    )
    SELECT 
        symbol,
        exchange,
        MAX(ABS(total_oi - prev_oi)) as max_daily_oi_change
    FROM DailyOI
    WHERE prev_oi IS NOT NULL
    GROUP BY symbol, exchange
    ORDER BY max_daily_oi_change DESC
    LIMIT 10;
"""
print("--- 1. Top 10 Symbols by OI Change ---")
display(conn.execute(query_1).df())

--- 1. Top 10 Symbols by OI Change ---


,SYMBOL,exchange,max_daily_oi_change
0,IDEA,NSE,349608000.0
1,YESBANK,NSE,101435400.0
2,SBIN,NSE,74850000.0
3,RELIANCE,NSE,59653000.0
4,INFY,NSE,59565600.0
5,INFY,BSE,58531200.0
6,IDFCFIRSTB,NSE,57960000.0
7,BHEL,NSE,54615000.0
8,GOLD,MCX,54432000.0
9,GMRINFRA,NSE,53505000.0


7-Day Rolling Volatility for NIFTY Options

In [14]:
query_2 = """
    SELECT 
        strptime(timestamp, '%d-%b-%Y')::DATE as trade_date,
        strike_pr,
        option_typ,
        close,
        STDDEV(close) OVER (
            PARTITION BY strike_pr, option_typ 
            ORDER BY strptime(timestamp, '%d-%b-%Y')::DATE 
            ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
        ) as rolling_7d_stddev
    FROM raw_trades
    WHERE symbol = 'NIFTY' AND instrument LIKE '%OPT%'
    ORDER BY strike_pr, option_typ, strptime(timestamp, '%d-%b-%Y')::DATE
    LIMIT 10;
"""
print("\n--- 2. 7-Day Rolling Volatility for NIFTY Options ---")
display(conn.execute(query_2).df())


--- 2. 7-Day Rolling Volatility for NIFTY Options ---


,trade_date,STRIKE_PR,OPTION_TYP,CLOSE,rolling_7d_stddev
0,2019-08-01,4600.0,CE,3955.65,NaN
1,2019-08-01,4600.0,CE,3683.20,136.225833
2,2019-08-01,4600.0,CE,3820.25,95.742258
3,2019-08-02,4600.0,CE,3683.20,121.844089
4,2019-08-02,4600.0,CE,3955.65,113.876260
5,2019-08-02,4600.0,CE,3820.25,111.228267
6,2019-08-05,4600.0,CE,3683.20,122.610773
7,2019-08-05,4600.0,CE,3955.65,122.610773
8,2019-08-05,4600.0,CE,3820.25,103.133156
9,2019-08-06,4600.0,CE,3955.65,122.523446


Cross-Exchange Comparison (MCX Gold vs NSE Futures)

In [15]:
query_3 = """
    SELECT 
        strptime(timestamp, '%d-%b-%Y')::DATE as trade_date,
        AVG(CASE WHEN exchange = 'MCX' AND symbol = 'GOLD' THEN close END) as avg_gold_mcx_settle,
        AVG(CASE WHEN exchange = 'NSE' AND instrument LIKE '%FUT%' THEN close END) as avg_eq_index_nse_settle
    FROM raw_trades
    GROUP BY strptime(timestamp, '%d-%b-%Y')::DATE
    HAVING AVG(CASE WHEN exchange = 'MCX' AND symbol = 'GOLD' THEN close END) IS NOT NULL
    ORDER BY strptime(timestamp, '%d-%b-%Y')::DATE
    LIMIT 10;
"""
print("\n--- 3. Cross-Exchange Comparison (MCX Gold vs NSE Futures) ---")
display(conn.execute(query_3).df())


--- 3. Cross-Exchange Comparison (MCX Gold vs NSE Futures) ---


,trade_date,avg_gold_mcx_settle,avg_eq_index_nse_settle
0,2019-08-01,93.890000,1882.931979
1,2019-08-02,59.476923,1889.637500
2,2019-08-05,162.294444,1873.796138
3,2019-08-06,153.360000,1897.701046
4,2019-08-07,106.486538,1877.593021
5,2019-08-08,84.073214,1897.370937
6,2019-08-09,90.104545,1914.375521
7,2019-08-13,100.768182,1877.774271
8,2019-08-14,104.192857,1890.743854
9,2019-08-16,146.039583,1896.716701


Option Chain Summary (Implied Volume)

In [17]:
query_4 = """
    SELECT 
        strptime(expiry_dt, '%d-%b-%Y')::DATE as expiry_date,
        strike_pr,
        SUM(CASE WHEN option_typ = 'CE' THEN CONTRACTS ELSE 0 END) as ce_implied_volume,
        SUM(CASE WHEN option_typ = 'PE' THEN CONTRACTS ELSE 0 END) as pe_implied_volume,
        SUM(CONTRACTS) as total_implied_volume
    FROM raw_trades
    WHERE option_typ IN ('CE', 'PE')
    GROUP BY strptime(expiry_dt, '%d-%b-%Y')::DATE, strike_pr
    ORDER BY strptime(expiry_dt, '%d-%b-%Y')::DATE, strike_pr
    LIMIT 10;
"""
print("\n--- 4. Option Chain Summary (Implied Volume) ---")
display(conn.execute(query_4).df())


--- 4. Option Chain Summary (Implied Volume) ---


,expiry_date,STRIKE_PR,ce_implied_volume,pe_implied_volume,total_implied_volume
0,2019-08-01,9600.0,0.0,10.0,10.0
1,2019-08-01,9650.0,0.0,0.0,0.0
2,2019-08-01,9700.0,0.0,63.0,63.0
3,2019-08-01,9750.0,0.0,0.0,0.0
4,2019-08-01,9800.0,0.0,16.0,16.0
5,2019-08-01,9850.0,0.0,0.0,0.0
6,2019-08-01,9900.0,0.0,5.0,5.0
7,2019-08-01,9950.0,0.0,41.0,41.0
8,2019-08-01,10000.0,19.0,1394.0,1413.0
9,2019-08-01,10050.0,0.0,285.0,285.0


Max Volume in Last 30 Days

In [18]:
query_5 = """
    WITH RankedVolume AS (
        SELECT 
            symbol,
            CONTRACTS as volume_contracts,
            strptime(timestamp, '%d-%b-%Y')::DATE as trade_date,
            RANK() OVER (PARTITION BY symbol ORDER BY CONTRACTS DESC) as vol_rank
        FROM raw_trades
        WHERE strptime(timestamp, '%d-%b-%Y')::DATE >= (SELECT MAX(strptime(timestamp, '%d-%b-%Y')::DATE) - INTERVAL 30 DAY FROM raw_trades)
    )
    SELECT 
        symbol,
        volume_contracts as max_volume,
        trade_date
    FROM RankedVolume
    WHERE vol_rank = 1
    ORDER BY max_volume DESC
    LIMIT 10;
"""
print("\n--- 5. Max Volume in Last 30 Days ---")
display(conn.execute(query_5).df())


# Make sure to also update the EXPLAIN ANALYZE execution for Query 5 [cite: 20]
explain_query = f"EXPLAIN ANALYZE {query_5}"
print("\n--- 6. EXPLAIN ANALYZE Output ---")
for row in conn.execute(explain_query).fetchall():
    print(row[0])
    print(row[1])


--- 5. Max Volume in Last 30 Days ---


,SYMBOL,max_volume,trade_date
0,BANKNIFTY,4564524,2019-11-14
1,NIFTY,1274605,2019-11-07
2,YESBANK,215823,2019-10-31
3,INFY,84779,2019-10-22
4,SBIN,83986,2019-10-25
5,BHARTIARTL,66343,2019-10-24
6,TATAMOTORS,64754,2019-10-29
7,IBULHSGFIN,62762,2019-10-17
8,RELIANCE,62144,2019-10-29
9,ICICIBANK,54549,2019-11-08



--- 6. EXPLAIN ANALYZE Output ---
analyzed_plan
┌─────────────────────────────────────┐
│┌───────────────────────────────────┐│
││    Query Profiling Information    ││
│└───────────────────────────────────┘│
└─────────────────────────────────────┘
EXPLAIN ANALYZE      WITH RankedVolume AS (         SELECT              symbol,             CONTRACTS as volume_contracts,             strptime(timestamp, '%d-%b-%Y')::DATE as trade_date,             RANK() OVER (PARTITION BY symbol ORDER BY CONTRACTS DESC) as vol_rank         FROM raw_trades         WHERE strptime(timestamp, '%d-%b-%Y')::DATE >= (SELECT MAX(strptime(timestamp, '%d-%b-%Y')::DATE) - INTERVAL 30 DAY FROM raw_trades)     )     SELECT          symbol,         volume_contracts as max_volume,         trade_date     FROM RankedVolume     WHERE vol_rank = 1     ORDER BY max_volume DESC     LIMIT 10; 
┌────────────────────────────────────────────────┐
│┌──────────────────────────────────────────────┐│
││              Total Time: 0.18

In [19]:
query_1_alternative = """
    WITH DailySymbolChange AS (
        SELECT 
            symbol,
            exchange,
            strptime(timestamp, '%d-%b-%Y')::DATE as trade_date,
            SUM(CHG_IN_OI) as total_daily_chg
        FROM raw_trades
        GROUP BY symbol, exchange, strptime(timestamp, '%d-%b-%Y')::DATE
    )
    SELECT 
        symbol,
        exchange,
        MAX(ABS(total_daily_chg)) as max_daily_oi_change
    FROM DailySymbolChange
    GROUP BY symbol, exchange
    ORDER BY max_daily_oi_change DESC
    LIMIT 10;
"""
print("--- 1. Top 10 Symbols by OI Change (Using CHG_IN_OI) ---")
display(conn.execute(query_1_alternative).df())

--- 1. Top 10 Symbols by OI Change (Using CHG_IN_OI) ---


,SYMBOL,exchange,max_daily_oi_change
0,IDEA,NSE,140252000.0
1,YESBANK,NSE,62873800.0
2,IDFCFIRSTB,NSE,57960000.0
3,INFY,NSE,42867600.0
4,ASHOKLEY,NSE,40214000.0
5,BHEL,NSE,32415000.0
6,SBIN,NSE,32328000.0
7,GMRINFRA,NSE,28395000.0
8,PNB,NSE,27916000.0
9,HDFCBANK,NSE,26834500.0


In [24]:
# ---------------------------------------------------------
# OPTIMIZATION REQUIREMENT: EXPLAIN ANALYZE
# ---------------------------------------------------------
# We will run EXPLAIN ANALYZE on the 30-day max volume query 
# to show how the database calculates the execution plan.

explain_query = """
    EXPLAIN ANALYZE
    WITH RankedVolume AS (
        SELECT 
            symbol,
            CONTRACTS as volume_contracts,
            strptime(timestamp, '%d-%b-%Y')::DATE as trade_date,
            RANK() OVER (PARTITION BY symbol ORDER BY CONTRACTS DESC) as vol_rank
        FROM raw_trades
        WHERE strptime(timestamp, '%d-%b-%Y')::DATE >= (SELECT MAX(strptime(timestamp, '%d-%b-%Y')::DATE) - INTERVAL 30 DAY FROM raw_trades)
    )
    SELECT 
        symbol,
        volume_contracts as max_volume,
        trade_date
    FROM RankedVolume
    WHERE vol_rank = 1;
"""

print("--- EXPLAIN ANALYZE Output for Max Volume Query ---")

# DuckDB returns the EXPLAIN plan as a table, so we format it for readability
result = conn.execute(explain_query).fetchall()
for row in result:
    print(row[0])  # Prints the step type
    print(row[1])  # Prints the detailed execution plan and timing

--- EXPLAIN ANALYZE Output for Max Volume Query ---
analyzed_plan
┌─────────────────────────────────────┐
│┌───────────────────────────────────┐│
││    Query Profiling Information    ││
│└───────────────────────────────────┘│
└─────────────────────────────────────┘
     EXPLAIN ANALYZE     WITH RankedVolume AS (         SELECT              symbol,             CONTRACTS as volume_contracts,             strptime(timestamp, '%d-%b-%Y')::DATE as trade_date,             RANK() OVER (PARTITION BY symbol ORDER BY CONTRACTS DESC) as vol_rank         FROM raw_trades         WHERE strptime(timestamp, '%d-%b-%Y')::DATE >= (SELECT MAX(strptime(timestamp, '%d-%b-%Y')::DATE) - INTERVAL 30 DAY FROM raw_trades)     )     SELECT          symbol,         volume_contracts as max_volume,         trade_date     FROM RankedVolume     WHERE vol_rank = 1; 
┌────────────────────────────────────────────────┐
│┌──────────────────────────────────────────────┐│
││              Total Time: 0.224s              ││
│└